# Product mapping update 2026-11-13: DPP-IV in the patient journey

**Why.** Decision X5 (3.1): the patient journey had DPP-IV switched off, so 220 NB04 stage x drug columns (and 9 of the 752 model features) could not be built. The user decided on 2026-11-13 to keep GLP-1 as its own category and to add DPP-IV as its own category.

**What changes (Rx basket only).** Input `Kerendia_Rx_Final_Product_Mapping_2026_11_06.csv`, output `Kerendia_Rx_Final_Product_Mapping_2026_11_13.csv` (3.1 volume, `final_product_mapping/`). The 2026_11_06 file is not touched. Px / Dx / Lab baskets are unchanged (the config picks the latest date per type).

| Change | Rows | Values |
| --- | --- | --- |
| Switch on the existing gliptin rows | 16 | `ACTIVE_JOURNEY = Y`, `TREATMENT_CATEGORY = DPP-IV`, `BRAND_LABEL = DPP-IV` |
| Add the 3.0 DPP-IV products missing from the basket | 5 (ONGLYZA, KOMBIGLYZE XR, NESINA, KAZANO, OSENI) | Same values, `PRODUCT_GROUP = OTHERS`, `IN_752_FEATURES = NO` |
| Not changed | GLYXAMBI, STEGLUJAN, QTERN (SGLT2, active), TRIJARDY XR (off) | |

**PATIENTS_TOTAL of the 5 new rows is left empty.** The values come from Sehar's original event list, which has no stated window (Dx strategy file). They could not be reproduced from the 3.1 pulls (tested all claims / new prescriptions over 1, 3, 6, 12 months: none matches), so no number is written rather than an estimate. The column is reference only; the pipeline only checks that it is a whole number or empty.

**Code changes that go with this file** (same date): `00_cohort_config` `HCP_RX_CATEGORY_MAP` gets `"DPP-IV": "Other- glucose mgmt"` (NB01 / NB04 Rx input keep the 3.0 structure, no duplicate column names); NB04 stage x drug step counts GLP-1 and DPP-IV only through the brand (no double count, no duplicate columns).

In [0]:
import pandas as pd, hashlib, os

FPM = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/final_product_mapping/"
OLD_DATE, NEW_DATE = "2026_11_06", "2026_11_13"
SRC = FPM + f"Kerendia_Rx_Final_Product_Mapping_{OLD_DATE}.csv"
DST = FPM + f"Kerendia_Rx_Final_Product_Mapping_{NEW_DATE}.csv"
CHANGE = f"3.1 change {NEW_DATE.replace('_', '-')}"
WHY = "DPP-IV own category in the patient journey (decision X5 revised 2026-11-13; 3.0 brand group DPP-IV)"

# The 16 gliptin rows already in the basket (all inactive today)
DPP_SWITCH_ON = ["JANUVIA", "TRADJENTA", "JANUMET", "JANUMET XR", "SITAGLIPTIN PHOSPHATE", "SAXAGLIPTIN HYDROCHLORIDE",
                 "ALOGLIPTIN", "JENTADUETO", "SITAGLIPTIN PHOSPHATE/MET", "SITAGLIPTIN", "JENTADUETO XR", "ZITUVIO",
                 "ALOGLIPTIN/METFORMIN HYDR", "ALOGLIPTIN/METFORMIN HCL", "SITAGLIPTIN/METFORMIN HYD", "ALOGLIPTIN/PIOGLITAZONE"]
# 3.0 DPP-IV products (Rx_01_new CATEGORY = 'DPP-IV') that are not in the basket
DPP_ADD = ["ONGLYZA", "KOMBIGLYZE XR", "NESINA", "KAZANO", "OSENI"]

md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
src_md5 = md5(SRC)
assert not os.path.exists(DST), f"{DST} already exists"

rx_old = pd.read_csv(SRC, dtype=str, keep_default_na=False)
rx_new = rx_old.copy()

def add_note(old, new):
    return f"{old}; {CHANGE}: {new}" if old else f"{CHANGE}: {new}"

for name in DPP_SWITCH_ON:
    i = rx_new.index[(rx_new.MATCH_TYPE == "NAME") & (rx_new.MATCH_VALUE == name) & (rx_new.STRENGTH == "")]
    assert len(i) == 1, f"expected one row for {name}, found {len(i)}"
    r = rx_new.loc[i[0]]
    assert r.ACTIVE_JOURNEY == "N" and r.TREATMENT_CATEGORY == "" and r.BRAND_LABEL == "", f"{name} is not an inactive row"
    rx_new.loc[i[0], ["ACTIVE_JOURNEY", "TREATMENT_CATEGORY", "BRAND_LABEL"]] = ["Y", "DPP-IV", "DPP-IV"]
    rx_new.loc[i[0], "NOTE"] = add_note(r.NOTE, f"N -> Y, DPP-IV. {WHY}")

assert not set(DPP_ADD) & set(rx_new.MATCH_VALUE), "a new DPP-IV name is already in the basket"
new_rows = pd.DataFrame([dict(MATCH_TYPE="NAME", MATCH_VALUE=n, CLUB_ID="", PRODUCT_GROUP="OTHERS", TREATMENT_CATEGORY="DPP-IV",
                              BRAND_LABEL="DPP-IV", STRENGTH="", CONDITION="", ACTIVE_JOURNEY="Y", PATIENTS_TOTAL="",
                              IN_752_FEATURES="NO",
                              NOTE=f"{CHANGE}: added, DPP-IV. {WHY}; product in the 3.0 DPP-IV list, missing from the basket. "
                                   "PATIENTS_TOTAL empty: the original event-list count cannot be reproduced")
                         for n in DPP_ADD])[rx_old.columns]
rx_new = pd.concat([rx_new, new_rows], ignore_index=True)

# ---- Checks before writing ----
assert len(rx_old) == 367 and len(rx_new) == 372, (len(rx_old), len(rx_new))
assert (rx_old.ACTIVE_JOURNEY == "Y").sum() == 245 and (rx_new.ACTIVE_JOURNEY == "Y").sum() == 266
assert (rx_new.TREATMENT_CATEGORY == "DPP-IV").sum() == 21 and (rx_new.BRAND_LABEL == "DPP-IV").sum() == 21
changed = [i for i in range(len(rx_old)) if not rx_old.iloc[i].equals(rx_new.iloc[i])]
assert sorted(rx_new.MATCH_VALUE.iloc[changed]) == sorted(DPP_SWITCH_ON), "rows other than the 16 gliptin rows changed"
assert not rx_new[rx_new.STRENGTH == ""].MATCH_VALUE.duplicated().any(), "product name on more than one row"

rx_new.to_csv(DST, index=False)
assert md5(SRC) == src_md5, "the 2026_11_06 file changed"
back = pd.read_csv(DST, dtype=str, keep_default_na=False)
assert back.equals(rx_new), "written file differs from the table"
print(f"Written {DST}: {len(back)} rows, {(back.ACTIVE_JOURNEY == 'Y').sum()} active, "
      f"{(back.TREATMENT_CATEGORY == 'DPP-IV').sum()} DPP-IV; changed rows {len(changed)}, added {len(new_rows)}")
display(back[back.TREATMENT_CATEGORY == "DPP-IV"])